# Arabizi keyboard: LoRA fine-tuning (P4)

Fine-tunes the two base models chosen in ADR 0004 (**Llama 3.2 1B** and **Qwen3-1.7B**) on the training rows exported with `pairs export-train`, then converts each to **GGUF Q4_K_M** for the Mac. Step-by-step guide: `docs/TRAINING.md` in the repository.

**Before running:** Settings → Accelerator → **GPU** (T4 or P100) · Settings → **Internet on** · Add data → your dataset with the export (`export.json`, `train-*.jsonl`, `val-*.jsonl`).

**Data:** since ADR 0028 every training row is public, so this notebook and its dataset may be public. Dev and test are never in the export: accuracy is measured on the Mac.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU keeps training simple and reproducible

In [ ]:
# Fresh copy every run: re-running in the same session would otherwise keep the old clone (git clone refuses an
# existing folder) and run outdated code.
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt

In [ ]:
import glob
import json

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
OUT = "/kaggle/working/out"

## 1. Llama 3.2 1B

In [ ]:
!python /kaggle/working/repo/train/finetune_lora.py --base llama3.2-1b --data "$DATA" --out "$OUT"

In [ ]:
run = max(glob.glob(f"{OUT}/*-llama3.2-1b-lora"))  # latest run
!bash /kaggle/working/repo/train/export_gguf.sh "{run}/merged" arabizi-llama3.2-1b-v1 "{run}"
!rm -rf "{run}/merged" "{run}/checkpoints"

## 2. Qwen3-1.7B

In [ ]:
!python /kaggle/working/repo/train/finetune_lora.py --base qwen3-1.7b --data "$DATA" --out "$OUT"

In [ ]:
run = max(glob.glob(f"{OUT}/*-qwen3-1.7b-lora"))  # latest run
!bash /kaggle/working/repo/train/export_gguf.sh "{run}/merged" arabizi-qwen3-1.7b-v1 "{run}"
!rm -rf "{run}/merged" "{run}/checkpoints"

## 3. Download

From the **Output** panel, download for each run: `arabizi-*-Q4_K_M.gguf`, its `.sha256` file and `run.json`. On the Mac, put the `.gguf` files in `models/` and the `run.json` files in `train/runs/`, then check the hashes and add the models to `config/models.json`, then score them on dev (docs/TRAINING.md).

In [ ]:
!find "$OUT" -maxdepth 2 -type f -printf "%10s  %p\n"